# 04 — EfficientNet-B0 Brand Classifier
## Brand Vision — Identical to Category Notebook

Only differences from `03_train_efficientnet_category.ipynb`:
- `DATA_DIR` points to `brand_logo_split/` (cropped logo images)
- `SAVE_DIR` points to `runs/brand/`
- Output: 50 brand classes instead of 20 categories
- **Target:** val accuracy ≥ 0.60 (ideally ≥ 0.75)

In [1]:
# Cell 1 — Imports
import os, json
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.amp import GradScaler, autocast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ON_KAGGLE  = os.path.exists("/kaggle")
BATCH_SIZE = 64 if ON_KAGGLE else 32

print(f"Device: {device} | Batch: {BATCH_SIZE}")

Device: cuda | Batch: 32


In [2]:
# Cell 2 — Config  ← only change from category notebook
DATA_DIR = Path("datasets/brand_logo_split")    # cropped logo images
SAVE_DIR = Path("runs/brand")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Cell 3 — Transforms (same as category notebook)
train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(str(DATA_DIR / "train"), train_tfm)
val_ds   = datasets.ImageFolder(str(DATA_DIR / "val"),   val_tfm)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

NUM_CLASSES = len(train_ds.classes)
print(f"Brand classes ({NUM_CLASSES}): {train_ds.classes}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Brand classes (15): ['ASUS', 'Apple', 'Armani', 'Bose', 'Canon', 'Casio', 'Converse', 'Lacoste', 'LouisVuitton', 'Oakley', 'RalphLauren', 'TheNorthFace', 'Timberland', 'TommyHilfiger', 'UnderArmour']
Train: 970 | Val: 311


In [4]:
# Cell 4 — Model
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
for p in model.features.parameters():
    p.requires_grad = False

in_feats = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_feats, NUM_CLASSES),
)
model = model.to(device)

In [5]:
# Cell 5 — Training loop
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler    = GradScaler("cuda")
best_acc  = 0.0
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

def run_epoch(loader, train: bool) -> tuple[float, float]:
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
                with autocast("cuda"):
                    out  = model(imgs)
                    loss = criterion(out, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                with autocast("cuda"):
                    out  = model(imgs)
                    loss = criterion(out, labels)
            total_loss += loss.item()
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total

# Phase 1 — head warmup
print("── Phase 1: Head warmup ──")
for epoch in range(5):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    vl_loss, vl_acc = run_epoch(val_dl,   train=False)
    print(f"Ep {epoch+1:02d} | tr_acc={tr_acc:.4f} | vl_acc={vl_acc:.4f}")

── Phase 1: Head warmup ──
Ep 01 | tr_acc=0.3320 | vl_acc=0.6238
Ep 02 | tr_acc=0.5866 | vl_acc=0.7588
Ep 03 | tr_acc=0.6753 | vl_acc=0.7781
Ep 04 | tr_acc=0.7351 | vl_acc=0.7878
Ep 05 | tr_acc=0.7515 | vl_acc=0.8071


In [6]:
# Cell 6 — Phase 2: Fine-tune
for p in model.features.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print("── Phase 2: Fine-tuning ──")
for epoch in range(30):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    vl_loss, vl_acc = run_epoch(val_dl,   train=False)
    scheduler.step()
    print(f"Ep {epoch+1:02d} | tr_acc={tr_acc:.4f} | vl_acc={vl_acc:.4f}")

    if vl_acc > best_acc:
        best_acc = vl_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_acc": vl_acc,
            "classes": train_ds.classes,
        }, str(SAVE_DIR / "best_model.pt"))
        print(f"  ✅ Saved (val_acc={vl_acc:.4f})")

status = "✅" if best_acc >= 0.60 else "⚠"
print(f"\n{status} Best val accuracy: {best_acc:.4f} (target ≥0.60)")

── Phase 2: Fine-tuning ──
Ep 01 | tr_acc=0.8134 | vl_acc=0.8682
  ✅ Saved (val_acc=0.8682)
Ep 02 | tr_acc=0.8918 | vl_acc=0.8842
  ✅ Saved (val_acc=0.8842)
Ep 03 | tr_acc=0.9165 | vl_acc=0.9228
  ✅ Saved (val_acc=0.9228)
Ep 04 | tr_acc=0.9608 | vl_acc=0.9293
  ✅ Saved (val_acc=0.9293)
Ep 05 | tr_acc=0.9577 | vl_acc=0.9325
  ✅ Saved (val_acc=0.9325)
Ep 06 | tr_acc=0.9567 | vl_acc=0.9486
  ✅ Saved (val_acc=0.9486)
Ep 07 | tr_acc=0.9732 | vl_acc=0.9486
Ep 08 | tr_acc=0.9742 | vl_acc=0.9550
  ✅ Saved (val_acc=0.9550)
Ep 09 | tr_acc=0.9856 | vl_acc=0.9486
Ep 10 | tr_acc=0.9928 | vl_acc=0.9550
Ep 11 | tr_acc=0.9897 | vl_acc=0.9550
Ep 12 | tr_acc=0.9928 | vl_acc=0.9614
  ✅ Saved (val_acc=0.9614)
Ep 13 | tr_acc=0.9969 | vl_acc=0.9614
Ep 14 | tr_acc=0.9969 | vl_acc=0.9550
Ep 15 | tr_acc=0.9969 | vl_acc=0.9550
Ep 16 | tr_acc=0.9969 | vl_acc=0.9518
Ep 17 | tr_acc=1.0000 | vl_acc=0.9550
Ep 18 | tr_acc=0.9969 | vl_acc=0.9550
Ep 19 | tr_acc=0.9959 | vl_acc=0.9582
Ep 20 | tr_acc=0.9979 | vl_acc=0.95

In [7]:
# Cell 7 — Export to ONNX
checkpoint = torch.load(str(SAVE_DIR / "best_model.pt"), weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval().cpu()

dummy     = torch.randn(1, 3, 224, 224)
onnx_path = str(SAVE_DIR / "brand_classifier.onnx")

torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

labels_path = str(SAVE_DIR / "brand_classifier_labels.json")
with open(labels_path, "w") as f:
    json.dump(train_ds.classes, f, indent=2)

print(f"ONNX saved:   {onnx_path}")
print(f"Labels saved: {labels_path}")
print("Next: copy both files to backend/weights/ then run export_onnx.py to verify")

C:\Users\mailm\AppData\Local\Temp\ipykernel_26240\3503938873.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0324 23:09:29.022000 26240 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0324 23:09:30.041000 26240 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\mailm\AppData\Local\Python\pythoncore-3.14-64\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Projects\INT428 DL CA01\brand-vision\venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "C:\Projects\INT428 DL CA01\brand-vision\venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Projects\INT428 DL CA01\brand-vision\venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Projects\INT428 DL CA01\brand-vision\venv\Lib\site-packages\onnx\version_conve

[torch.onnx] Optimize the ONNX graph...
Applied 99 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅
ONNX saved:   runs\brand\brand_classifier.onnx
Labels saved: runs\brand\brand_classifier_labels.json
Next: copy both files to backend/weights/ then run export_onnx.py to verify
